In [ ]:
# Upgrade PyTorch to a unified CUDA 12.1 version
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install and upgrade all Hugging Face and evaluation libraries together
!pip install --upgrade transformers datasets evaluate accelerate peft bitsandbytes sentencepiece rouge_score bert_score nltk

In [ ]:
import os
import gc
import json
import glob
import torch
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import FastLanguageModel


# Clear out VRAM fragmentation
gc.collect()
torch.cuda.empty_cache()

In [26]:
def migrate_essay_data(folder_path):
    all_essay_records = []
    search_pattern = os.path.join(folder_path, "*_essay.jsonl")
    essay_files = glob.glob(search_pattern)
    
    print(f"Scanning for data files in: {folder_path}")
    print(f"Found {len(essay_files)} matching .jsonl files.")
    
    for file_path in essay_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    record = json.loads(line)
                    if record.get("type") == "essay" and record.get("model_answer"):
                        all_essay_records.append(record)
                except Exception:
                    continue
                    
    print(f"-> Successfully extracted {len(all_essay_records)} valid essay records.")
    return all_essay_records

def tokenize_function(examples):
    tokenized_inputs = {"input_ids": [], "attention_mask": [], "labels": []}
    
    for c, s, q, a in zip(examples["context"], examples["scenario_context"], examples["question"], examples["model_answer"]):
        full_text = f"Context: {c}\n\nScenario: {s}\nQuestion: {q}\nAnswer: {a}{tokenizer.eos_token}"
        
        tokens = tokenizer(
            full_text,
            max_length=MAX_SEQ_LENGTH,
            truncation=True,
            padding=False, 
        )
        
        input_ids = tokens["input_ids"]
        attention_mask = tokens["attention_mask"]
        
        labels = input_ids.copy()
        
        tokenized_inputs["input_ids"].append(input_ids)
        tokenized_inputs["attention_mask"].append(attention_mask)
        tokenized_inputs["labels"].append(labels)
        
    return tokenized_inputs

def generate_synthetic_data(context_text):
    eval_prompt = f"Context: {context_text}\n\nScenario:"
    inputs = tokenizer([eval_prompt], return_tensors = "pt").to("cuda")
    
    outputs = model.generate(
        **inputs, 
        max_new_tokens = 350, 
        use_cache = True,
        temperature = 0.8,    
        do_sample = True,
        eos_token_id = tokenizer.eos_token_id
    )
    
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_text

In [ ]:
MODEL_NAME = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit" 
DATA_FOLDER = "/kaggle/input/datasets/adhamashraf202200953/technical-parsed-questions"
OUTPUT_DIR = "./outputs/UNSLOTH_PHI3"

MAX_SEQ_LENGTH = 512
BATCH_SIZE = 2
GRADIENT_ACC_STEPS = 16

In [ ]:
raw_essay_data = migrate_essay_data(DATA_FOLDER)
if len(raw_essay_data) == 0:
    raise ValueError("CRITICAL: Zero records found! Check your dataset path.")

dataset = Dataset.from_list(raw_essay_data)

print("Loading model via Unsloth optimized memory kernels...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,             # Auto-detects float16/bfloat16
    load_in_4bit = True,      
    device_map = "auto",
)

# Apply specialized LoRA architectural patching
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,         
    bias = "none",
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
)

# Explicitly ensure padding is synchronized to prevent alignment errors
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# Process the dataset and discard all original raw text string columns
tokenized_dataset = dataset.map(
    tokenize_function, 
    batched=True, 
    remove_columns=dataset.column_names
)
split_dataset = tokenized_dataset.train_test_split(test_size=0.1)

print(f"Training Dataset Samples: {len(split_dataset['train'])}")

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = split_dataset["train"],
    eval_dataset = split_dataset["test"],
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing = False,
    # This specific data collator ensures perfect label/input alignment on every batch
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True),
    args = TrainingArguments(
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRADIENT_ACC_STEPS,
        warmup_steps = 5,
        max_steps = 60,            
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = OUTPUT_DIR,
        report_to = "none",        
    ),
)

In [23]:
print("Starting training loop...")
trainer_stats = trainer.train()

!zip -r "UNSLOTH-PHI3-FINETUNED.zip" "./outputs"

Scanning for data files in: /kaggle/input/datasets/adhamashraf202200953/technical-parsed-questions
Found 4 matching .jsonl files.
-> Successfully extracted 2209 valid essay records.
Loading model via Unsloth optimized memory kernels...
==((====))==  Unsloth 2026.5.2: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Map:   0%|          | 0/2209 [00:00<?, ? examples/s]

Training Dataset Samples: 1988
Starting training loop...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 1,988 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 16 x 1) = 32
 "-____-"     Trainable parameters = 29,884,416 of 3,850,963,968 (0.78% trained)


Step,Training Loss
5,2.187541
10,1.893195
15,1.625918
20,1.485231
25,1.503609
30,1.442777
35,1.411681
40,1.408251
45,1.387365
50,1.401561


Unsloth: Restored added_tokens_decoder metadata in ./outputs/UNSLOTH_PHI3/checkpoint-60/tokenizer_config.json.


tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in ./outputs/UNSLOTH_PHI3/checkpoint-60.


  adding: outputs/ (stored 0%)
  adding: outputs/UNSLOTH_PHI3/ (stored 0%)
  adding: outputs/UNSLOTH_PHI3/README.md (deflated 43%)
  adding: outputs/UNSLOTH_PHI3/checkpoint-60/ (stored 0%)
  adding: outputs/UNSLOTH_PHI3/checkpoint-60/README.md (deflated 65%)
  adding: outputs/UNSLOTH_PHI3/checkpoint-60/scaler.pt (deflated 64%)
  adding: outputs/UNSLOTH_PHI3/checkpoint-60/adapter_model.safetensors (deflated 7%)
  adding: outputs/UNSLOTH_PHI3/checkpoint-60/optimizer.pt (deflated 10%)
  adding: outputs/UNSLOTH_PHI3/checkpoint-60/adapter_config.json (deflated 58%)
  adding: outputs/UNSLOTH_PHI3/checkpoint-60/tokenizer_config.json (deflated 86%)
  adding: outputs/UNSLOTH_PHI3/checkpoint-60/rng_state.pth (deflated 26%)
  adding: outputs/UNSLOTH_PHI3/checkpoint-60/chat_template.jinja (deflated 60%)
  adding: outputs/UNSLOTH_PHI3/checkpoint-60/training_args.bin (deflated 53%)
  adding: outputs/UNSLOTH_PHI3/checkpoint-60/trainer_state.json (deflated 69%)
  adding: outputs/UNSLOTH_PHI3/checkpoin

In [25]:
print("\n--- Generating Fine-Tuned Phi-3 Evaluation Output ---")

FastLanguageModel.for_inference(model) 

# Your evaluation validation inputs
test_context = """
You are tasked with developing a monitoring system for your company's production containers. 
The system must continuously track the health of each container and take appropriate action if necessary. 
However, you've noticed that some containers are exiting immediately with status code 0, 
leading to unreliable monitoring data.
"""
# Generate and view the output
phi3_generated_answer = generate_synthetic_data(test_context)

print(f"Scenario: You are tasked with developing a monitoring system...")
print(f"Question: {test_question}\n")
print(f"Answer: {phi3_generated_answer}")

Both `max_new_tokens` (=350) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Generating Fine-Tuned Phi-3 Evaluation Output ---


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:254: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Scenario: You are tasked with developing a monitoring system...
Question: Explain how you would design your monitoring system to handle containers that exit immediately with status code 0. Be sure to address the tradeoffs between continuous monitoring and resource consumption.

Answer: Context: 
You are tasked with developing a monitoring system for your company's production containers. 
The system must continuously track the health of each container and take appropriate action if necessary. 
However, you've noticed that some containers are exiting immediately with status code 0, 
leading to unreliable monitoring data.


Scenario: Your monitoring system currently uses Docker's restart policy to handle container failures. 
You need to modify this approach to account for containers with status code 0, which may exit immediately.

Question: Propose an alternative approach to handling exiting containers with status code 0 in your monitoring system. Explain the tradeoffs between this approa